In [0]:
%pip install apache-sedona==1.7.1

  Obtaining dependency information for apache-sedona==1.7.1 from https://files.pythonhosted.org/packages/db/37/0e0ba5e6a1a85f98fdf470b166ff1f35ba63c0f733343b41dd951a63719e/apache_sedona-1.7.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for attrs from https://files.pythonhosted.org/packages/64/b4/17d4b0b2a2dc85a6df63d1157e028ed19f90d4cd97c36717afef2bc2f395/attrs-26.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for shapely>=1.7.0 from https://files.pythonhosted.org/packages/13/02/58b0b8d9c17c93ab6340edd8b7308c0c5a5b81f94ce65705819b7416dba5/shapely-2.1.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/206.6 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 204.8/206.6 kB 6.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.1 MB ? eta -:--

In [0]:
%restart_python

In [0]:
%pip install geopandas

  Obtaining dependency information for geopandas from https://files.pythonhosted.org/packages/3c/78/6a04792ace63a93e162f1305392d500ae8ddcb620e7eb88a22fd622b35bb/geopandas-1.1.3-py3-none-any.whl.metadata
  Obtaining dependency information for numpy>=1.24 from https://files.pythonhosted.org/packages/cf/c5/9fcb7e0e69cef59cf10c746b84f7d58b08bc66a6b7d459783c5a4f6101a6/numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for pyogrio>=0.7.2 from https://files.pythonhosted.org/packages/89/a4/0aef5837b4e11840f501e48e01c31242838476c4f4aff9c05e228a083982/pyogrio-0.12.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for pandas>=2.0.0 from https://files.pythonhosted.org/packages/20/17/ec40d981705654853726e7ac9aea9ddbb4a5d9cf54d8472222f4f3de06c2/pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/79.5 kB ? eta -:--:--
     ━

In [0]:
from pyspark.sql.functions import expr, first, broadcast, col, when, coalesce, lower, trim, lit
from sedona.spark import SedonaContext

# 셔플 파티션 설정 및 Sedona 초기화
spark.conf.set("spark.sql.shuffle.partitions", "200")
sedona = SedonaContext.create(spark)


# 반려견 산책 가능 도로 등급만 필터링
safe_highways = [
    "footway", "path", "pedestrian", "residential", 
    "living_street", "track", "steps"
]

# WKT → 공간 객체 변환
edges_clean = spark.table("bronze.osm_edges") \
    .filter(col("highway").isin(safe_highways)) \
    .withColumn("geom_e", expr("ST_GeomFromWKT(geometry)")) \
    .select("id", "length", col("u").alias("start_node"), col("v").alias("end_node"), 
            "surface", "smoothness", "highway", "geometry", "geom_e")

# 공간 조인 전 캐시 고정
edges_clean.cache()
edges_clean.count()
print("기준 도로 데이터 전처리 완료!")


# V-World 토양 5종 레이어 로드 및 공간 객체 변환
soil_type_clean  = spark.table("bronze.vworld_soil_type").select("DEEPSOIL", expr("ST_GeomFromWKT(geometry)").alias("geom_s"))
slope_clean      = spark.table("bronze.vworld_slope").select("SOILSLOPE", expr("ST_GeomFromWKT(geometry)").alias("geom_sl"))
gravel_clean     = spark.table("bronze.vworld_gravel").select("SUR_STON", expr("ST_GeomFromWKT(geometry)").alias("geom_g"))
soil_depth_clean = spark.table("bronze.vworld_soil_depth").select("VLDSOILDEP", expr("ST_GeomFromWKT(geometry)").alias("geom_sd"))
drainage_clean   = spark.table("bronze.vworld_drainage").select("SOILDRA", expr("ST_GeomFromWKT(geometry)").alias("geom_d"))


# ST_Distance < 0.00005 기준 연쇄 공간 조인 (Broadcast로 셔플 비용 절감)
edges_joined = edges_clean \
    .join(broadcast(soil_type_clean),  expr("ST_Distance(geom_e, geom_s) < 0.00005"), "left") \
    .join(broadcast(slope_clean),      expr("ST_Distance(geom_e, geom_sl) < 0.00005"), "left") \
    .join(broadcast(gravel_clean),     expr("ST_Distance(geom_e, geom_g) < 0.00005"), "left") \
    .join(broadcast(soil_depth_clean), expr("ST_Distance(geom_e, geom_sd) < 0.00005"), "left") \
    .join(broadcast(drainage_clean),   expr("ST_Distance(geom_e, geom_d) < 0.00005"), "left")


# 중복 제거 — 여러 지형 매칭 시 첫 번째 유효값 채택
silver_df = edges_joined.groupBy(
    "id", "start_node", "end_node", "geometry", "length", "highway", "surface", "smoothness"
).agg(
    first("DEEPSOIL", True).alias("soil_type"),
    first("SOILSLOPE", True).alias("soil_slope"),
    first("SUR_STON", True).alias("gravel_content"),
    first("VLDSOILDEP", True).alias("soil_depth"),
    first("SOILDRA", True).alias("drainage_class")
)

# 잔여 NULL 처리 후 캐시 고정
silver_df = silver_df.withColumn("soil_type", coalesce(col("soil_type"), lit("Unknown")))

silver_df.cache()
print(f"✅ 데이터 통합 성공 (최종 유효 도로 수: {silver_df.count()})")
silver_df.show(10)

기준 도로 데이터 전처리 완료!
✅ 데이터 통합 성공 (최종 유효 도로 수: 143295)
+--------+-----------+-----------+--------------------+-------+-------------+-------+----------+---------+----------+--------------+----------+--------------+
|      id| start_node|   end_node|            geometry| length|      highway|surface|smoothness|soil_type|soil_slope|gravel_content|soil_depth|drainage_class|
+--------+-----------+-----------+--------------------+-------+-------------+-------+----------+---------+----------+--------------+----------+--------------+
|37399188| 2461164532|10082252612|LINESTRING (127.0...|  36.12|  residential|   NULL|      NULL|  Unknown|      NULL|          NULL|      NULL|          NULL|
|37399583| 5810221894|10009232095|LINESTRING (127.1...| 48.151|  residential|   NULL|      NULL|   사양질|      2-7%|           <10|     20-50|          양호|
|38233805| 5996195661| 8727059730|LINESTRING (127.0...|   3.65|  residential|asphalt|      NULL|   식양질|     7-15%|         10-35|    50-100|          양호|
|4048

In [0]:
from pyspark.sql.functions import expr, first, avg, col, when, trim, coalesce, lit, round, lower

# 집계 단계 — 파티션 수 축소 (200 → 64)
spark.conf.set("spark.sql.shuffle.partitions", "64")

# 경사도 범위 문자열 → 구간 중앙값으로 수치 변환
slope_numeric = when(trim(col("SOILSLOPE")) == "0-2%", 1.0) \
                .when(trim(col("SOILSLOPE")) == "2-7%", 4.5) \
                .when(trim(col("SOILSLOPE")) == "7-15%", 11.0) \
                .when(trim(col("SOILSLOPE")) == "15-30%", 22.5) \
                .when(trim(col("SOILSLOPE")) == "30-60%", 45.0) \
                .when(trim(col("SOILSLOPE")) == "60-100%", 80.0) \
                .otherwise(None)

# geometry를 groupBy에서 제외 — 셔플 데이터 볼륨 최소화
silver_df = edges_joined.withColumn("slope_val", slope_numeric).groupBy(
    "id", "start_node", "end_node"
).agg(
    first("geometry").alias("geometry"),
    first("length").alias("length"),
    first("surface").alias("surface"),
    first("smoothness").alias("smoothness"),
    first("highway").alias("highway"),
    first("DEEPSOIL", True).alias("soil_type"),
    avg("slope_val").alias("avg_slope_raw"),
    first("SUR_STON", True).alias("gravel_content"),
    first("VLDSOILDEP", True).alias("soil_depth"),
    first("SOILDRA", True).alias("drainage_class")
)

# 공간 조인 미매칭 구간 — highway 타입별 기본 경사도 부여
# steps는 수치 경사도 의미 없으므로 NULL 처리
silver_df = silver_df.withColumn(
    "avg_slope",
    when(col("highway") == "steps", lit(None))
    .when(col("avg_slope_raw").isNotNull(), col("avg_slope_raw"))
    .otherwise(
        when(col("highway") == "track", 7.0)
        .when(col("highway").isin("living_street", "path"), 3.5)
        .when(col("highway").isin("footway"), 2.5)
        .otherwise(1.5)
    )
).withColumn(
    # 평지(≤3%) / 완만(≤8%) / 급경사(>8%) / 계단 분류
    "slope_type",
    when(col("highway") == "steps", lit("계단"))
    .when(col("avg_slope").isNull(), lit("정보없음"))
    .when(col("avg_slope") <= 3.0, lit("평지"))
    .when(col("avg_slope") <= 8.0, lit("완만"))
    .otherwise(lit("급경사"))
)

# 소수점 정리 및 텍스트 정규화
silver_df = silver_df.withColumn("avg_slope", round(col("avg_slope"), 1)) \
                     .withColumn("surface", lower(trim(col("surface")))) \
                     .withColumn("soil_type", trim(col("soil_type"))) \
                     .drop("avg_slope_raw")

# 이후 Score 산출(Phase 3~5)을 위해 캐시 고정
silver_df.cache()

print(f" 경사도 데이터 정제 및 보정 성공 (최종 행수: {silver_df.count()})")

# 1. 데이터베이스 생성 (이미 있다면 무시)
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# 2. 저장 전 파티션 최적화 (선택 사항)
# 최종 행 수가 22만 건 정도라면, 파일 갯수를 적절히 조절해서 저장하는 게 나중에 Gold에서 부를 때 더 빠릅니다.
silver_df_final = silver_df.repartition(10) # 파일 10개 정도로 나눠서 저장

# 3. Delta 테이블로 저장
silver_df_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.walk_features")

print("Silver 레이어: silver.walk_features 저장 완료!")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col, when, lit, lower, coalesce, expr

# 1. ID 부여
leisure_with_id = spark.table("bronze.osm_leisure") \
    .withColumn("id", monotonically_increasing_id()) 

# 2. 키워드 패턴
EXCLUDE_KEYWORDS = ["아파트", "잠실주공5단지", "아크로힐스", "로렌하우스", "대치에스케이뷰", "래미안대치"]
pattern = "|".join(EXCLUDE_KEYWORDS)

# 3. 가공 작업 (pet 필터 제거 및 안전 장치 추가)
poi_silver = leisure_with_id.filter(
    col("leisure").isin("park", "dog_park") & 
    # (~col("pet").isin("no")) # 만약 pet 컬럼이 나중에 생길걸 대비한다면 이렇게 쓰지만, 지금은 아예 빼는게 안전합니다.
    (~coalesce(col("name"), lit("")).rlike(pattern))
).select(
    "id",
    when(col("name").isNull(), lit("장소명 없음")).otherwise(col("name")).alias("name"),
    "leisure",
    "geometry"
).withColumn("geom_p", expr("ST_GeomFromWKT(geometry)"))

# 4. 저장
poi_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.park")

print("✅ pet 컬럼 부재 이슈 해결 및 silver.park 테이블 생성 완료!")

In [0]:
walk_features = spark.table("silver.park")
walk_features.printSchema()